# Step 0: Explore listings.csv, choose columns, create the sample

Goal of this notebook (this is a notebook for data exploration):
1. Load the raw file and check that it is parsed correctly
2. Look at the schema and choose the columns we keep
3. Measure nulls and look at raw values, to fill in `data_contract.md`
4. Create a 5,000-row sample for fast work

Expected location: `notebooks/scratch/00_step0_explore.ipynb`, with the data in `data/raw/listings.csv`.

In [1]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("airbnb-step0")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

DATA_PATH = "../../data/raw/listings.csv"
SAMPLE_DIR = "../../data/sample"
SAMPLE_SIZE = 5000
SEED = 42

assert os.path.exists(DATA_PATH), f"File not found: {os.path.abspath(DATA_PATH)}"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/27 15:36:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Load and check parsing
Descriptions contain line breaks and doubled quotes, so `multiLine` and `escape` are required.
Everything is read as text on purpose: we decide the types ourselves during cleaning.

In [2]:
raw = (spark.read
       .option("header", True)
       .option("multiLine", True)
       .option("escape", '"')
       .csv(DATA_PATH))

n_rows = raw.count()
print(f"Rows: {n_rows:,} | Columns: {len(raw.columns)}")

# If parsing broke, pieces of descriptions end up in the id column.
bad_ids = raw.filter(~F.col("id").rlike("^[0-9]+$")).count()
print("Rows with a non-numeric id (should be 0):", bad_ids)

Rows: 77,149 | Columns: 90


[Stage 4:>                                                          (0 + 1) / 1]

Rows with a non-numeric id (should be 0): 0


## 2. All columns

In [3]:
for i, c in enumerate(raw.columns):
    print(f"{i:3d}  {c}")

  0  id
  1  listing_url
  2  scrape_id
  3  last_scraped
  4  source
  5  name
  6  description
  7  neighborhood_overview
  8  picture_url
  9  host_id
 10  host_url
 11  host_profile_id
 12  host_profile_url
 13  host_name
 14  host_since
 15  hosts_time_as_user_years
 16  hosts_time_as_user_months
 17  hosts_time_as_host_years
 18  hosts_time_as_host_months
 19  host_location
 20  host_about
 21  host_response_time
 22  host_response_rate
 23  host_acceptance_rate
 24  host_is_superhost
 25  host_thumbnail_url
 26  host_picture_url
 27  host_neighbourhood
 28  host_listings_count
 29  host_total_listings_count
 30  host_verifications
 31  host_has_profile_pic
 32  host_identity_verified
 33  neighbourhood
 34  neighbourhood_cleansed
 35  neighbourhood_group_cleansed
 36  latitude
 37  longitude
 38  property_type
 39  room_type
 40  accommodates
 41  bathrooms
 42  bathrooms_text
 43  bedrooms
 44  beds
 45  amenities
 46  price
 47  price_quote_checkin_date
 48  price_quote_checko

In [4]:
rows = raw.limit(200).collect()

for i, c in enumerate(raw.columns):
    examples = []
    for r in rows:
        v = r[c]
        if v is not None and v not in examples:
            examples.append(v)
        if len(examples) == 3:
            break
    shown = [str(v)[:60] for v in examples]
    print(f"{i:3d}  {c:45s} {shown}")

26/11/27 15:37:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  0  id                                            ['3109', '7397', '9952']
  1  listing_url                                   ['https://www.airbnb.com/rooms/3109', 'https://www.airbnb.com/rooms/7397', 'https://www.airbnb.com/rooms/9952']
  2  scrape_id                                     ['20260916052215']
  3  last_scraped                                  ['2026-09-19', '2026-09-22', '2026-09-21']
  4  source                                        ['previous scrape']
  5  name                                          ['zen and calm', 'MARAIS - 2ROOMS APT - 2/4 PEOPLE', 'Paris petit coin douillet']
  6  description                                   ['Lovely Appartment with one bedroom with a Queen size bed.<br', 'VERY CONVENIENT, WITH THE BEST LOCATION !', 'I am a retired lady, who offers a pleasant little apartment ']
  7  neighborhood_overview                         []
  8  picture_url                                   ['https://a0.muscache.com/pictures/miso/Hosting-3109/original/'

## 2.5 Columns that, at first sight, have no relation to the price

**URLs & Photos**
1. listing_url (1)
2. picture_url (8)
3. host_url (10)
4. host_profile_url (12)
5. host_thumbnail_url (25)
6. host_picture_url (26)

**System IDs and Registration numbers**
1. id
2. scrape_id
3. host_id
4. host_profile_id

**Scraping metadata**
1. last_scraped (3)
2. source (4)
3. calendar_updated (60)
4. calendar_last_scraped (66)
5. host_name (13)

**Free text: we are not using Natural Language Processing (NLP)**
1. name (5)
2. description (6)
3. neighborhood_overview (7) // This one is also empty
4. host_about (20)

**Variables that repeat existing data or are discarded at first**
1. host_location (19)  // Host location is already explained by latitude and longitude
2. license (83) // License doesn't seem to have a relationship because it is an index
3. first_review // timestamp
4. last_review // timestamp

In [5]:
NO_RELATION_COLS = [
    # URLs & Photos
    "listing_url",
    "picture_url",
    "host_url",
    "host_profile_url",
    "host_thumbnail_url",
    "host_picture_url",

    # System IDs & Registration numbers (remove "id" from this list if keeping it as a reference index)
    "id",
    "scrape_id",
    "host_id",
    "host_profile_id",

    # Scraping metadata & personal names
    "last_scraped",
    "source",
    "calendar_updated",
    "calendar_last_scraped",
    "host_name",

    # Free text (not using NLP)
    "name",
    "description",
    "neighborhood_overview",
    "host_about",

    # Variables that are 100% null => the value is None in all the records
    "host_verifications"  
]

**Null rate in all the columns**
These are the columns that have 100% null values.

In [6]:
# 0. Skip the "no relation" columns defined above (NO_RELATION_COLS)
cols_to_check = [c for c in raw.columns if c not in NO_RELATION_COLS]
print(f"Checking {len(cols_to_check)} of {len(raw.columns)} columns "
      f"({len(raw.columns) - len(cols_to_check)} skipped as no-relation)\n")

# 1. Compute exact null % across the full dataset in one Spark pass
#    (counts true nulls, empty strings, and literal "NULL" text)
null_exprs = [
    F.round(
        100 * F.avg(
            (
                F.col(c).isNull()
                | (F.trim(F.col(c)) == "")
                | (F.upper(F.trim(F.col(c))) == "NULL")
            ).cast("int")
        ),
        1,
    ).alias(c)
    for c in cols_to_check
]
null_rates = raw.agg(*null_exprs).collect()[0].asDict()

# 2. Collect up to 3 non-null examples per column from the first 200 rows
rows = raw.limit(200).collect()

print(f"{'#':>3s}  {'Column':45s} {'Null %':>6s}  Examples")
print("-" * 100)
for c in cols_to_check:
    i = raw.columns.index(c)  # keep the original column number
    examples = []
    for r in rows:
        v = r[c]
        if (
            v is not None
            and str(v).strip() != ""
            and str(v).strip().upper() != "NULL"
            and v not in examples
        ):
            examples.append(v)
        if len(examples) == 3:
            break
    shown = [str(v)[:50] for v in examples]
    print(f"{i:3d}  {c:45s} {null_rates[c]:5.1f}%  {shown}")

Checking 70 of 90 columns (20 skipped as no-relation)



  #  Column                                        Null %  Examples
----------------------------------------------------------------------------------------------------
 14  host_since                                    100.0%  []
 15  hosts_time_as_user_years                        0.0%  ['17', '18', '16']
 16  hosts_time_as_user_months                       0.0%  ['11', '0', '10']
 17  hosts_time_as_host_years                        0.0%  ['13', '15', '14']
 18  hosts_time_as_host_months                       0.0%  ['1', '8', '7']
 19  host_location                                  21.4%  ['Paris, France', 'France', 'Bidart, France']
 21  host_response_time                            100.0%  []
 22  host_response_rate                            100.0%  []
 23  host_acceptance_rate                          100.0%  []
 24  host_is_superhost                               0.0%  ['f', 't']
 27  host_neighbourhood                            100.0%  []
 28  host_listings_count              

## 2.6 Columns that are 100% null (completely empty)

**Host attributes**
1. host_since (14)
2. host_response_time (21)
3. host_response_rate (22)
4. host_acceptance_rate (23)
5. host_thumbnail_url (25)
6. host_neighbourhood (27)
7. host_total_listings_count (29)

**Location & Text**
1. neighborhood_overview (7)
2. neighbourhood (33)
3. neighbourhood_group_cleansed (35)

**Property & Calendar attributes**
1. bathrooms (41)
2. calendar_updated (60)
3. instant_bookable (84)

In [7]:
NULL_100_COLS = [
    "neighborhood_overview",
    "host_since",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_thumbnail_url",
    "host_neighbourhood",
    "host_total_listings_count",
    "neighbourhood",
    "neighbourhood_group_cleansed",
    "bathrooms",
    "calendar_updated",
    "instant_bookable",
]

ERASED_COLUMNS = list(dict.fromkeys(NULL_100_COLS + NO_RELATION_COLS))

In [8]:
# Drop the columns that are 100% null (NULL_100_COLS)

# 1. Only drop columns that actually exist in the DataFrame
missing = [c for c in ERASED_COLUMNS if c not in raw.columns]
if missing:
    print("Not found in the data (skipped):", missing)
to_drop = [c for c in ERASED_COLUMNS if c in raw.columns]

# 2. Safety check: confirm they are really 100% null before dropping
#    (same rule as the null analysis: null, empty string or literal "NULL")
check = raw.agg(*[
    F.round(100 * F.avg((
        F.col(c).isNull()
        | (F.trim(F.col(c)) == "")
        | (F.upper(F.trim(F.col(c))) == "NULL")
    ).cast("int")), 1).alias(c)
    for c in to_drop
]).collect()[0].asDict()

# Only warn for columns that are not 100% null AND are not in NO_RELATION_COLS
# (no-relation columns are dropped on purpose, even if they have data)
not_empty = {
    c: pct
    for c, pct in check.items()
    if pct < 100 and c not in NO_RELATION_COLS
}

if not_empty:
    print("WARNING: these columns are NOT 100% null, review them:", not_empty)

# 3. Drop them
df = raw.drop(*to_drop)

print(f"Number of ERASED columns: {len(NULL_100_COLS)}")
print(f"Dropped {len(to_drop)} columns")
print(f"Columns: {len(raw.columns)} -> {len(df.columns)}")

Number of ERASED columns: 13
Dropped 30 columns
Columns: 90 -> 60


### Drop Treatment Categories (PCA & Random Forest Pipeline)

Rather than applying a single rule to all variables with nulls above 0.0%, we group them into seven treatment categories based on their role in the PCA and Random Forest pipeline.

**1. Drop Rows (1 col)**
* Columns: 
    - **price** (46 — 43.6%)
* Why this treatment fits: It is the target variable (y): only rows with a price can be used for supervised training.

**2. Drop unrelated columns and columns that are 100% null**

* Columns 100% null (13):
    - **host_since** (14)
    - **host_response_time** (21)
    - **host_response_rate** (22)
    - **host_acceptance_rate** (23)
    - **host_thumbnail_url** (25)
    - **host_neighbourhood** (27)
    - **host_total_listings_count** (29)
    - **neighborhood_overview** (7)
    - **neighbourhood** (33)
    - **neighbourhood_group_cleansed** (35)
    - **bathrooms** (41)
    - **calendar_updated** (60)
    - **instant_bookable** (84)
* Unrelated columns (18 more):
    - URLs & Photos: **listing_url** (1), **picture_url** (8), **host_url** (10), **host_profile_url** (12), **host_picture_url** (26)
    - System IDs: **id**, **scrape_id**, **host_id**, **host_profile_id**
    - Scraping metadata & names: **last_scraped** (3), **source** (4), **calendar_last_scraped** (66), **host_name** (13)
    - Free text (no NLP): **name** (5), **description** (6), **host_about** (20)
      
**3. Drop the leakage columns related to the target, price (6 columns)**
* Columns:
    - **price_quote_checkin_date**
    - **price_quote_checkout_date**
    - **price_quote_total_price**
    - **price_quote_price_per_night**
    - **price_quote_raw**
    - **estimated_revenue_l365d**
* Why this treatment fits: Excluded on domain or structural grounds (target leakage) these variables would let the model cheat in the prediction, because they are already related to the price.
* Evidence:
    - **estimated_revenue_l365d** = estimated_occupancy_l365d × price on every row (the largest gap is 0.50 €, which is rounding). Keeping it would leak the price into the model.

In [9]:
LEAKAGE_COLS = [
    "price_quote_checkin_date",
    "price_quote_checkout_date",
    "price_quote_total_price",
    "price_quote_price_per_night",
    "price_quote_raw",
    "estimated_revenue_l365d",
]

In [10]:
TARGET_COL = "price"

# Combine exclusions without duplicates while preserving order.
# Never erase the target, even if it is listed in LEAKAGE_COLS.
ERASED_COLUMNS = [
    c for c in dict.fromkeys(NULL_100_COLS + NO_RELATION_COLS + LEAKAGE_COLS)
    if c != TARGET_COL
]

# Check for any excluded column names that are not in raw
missing_in_raw = [c for c in ERASED_COLUMNS if c not in raw.columns]
if missing_in_raw:
    print("Warning - columns not found in raw:", missing_in_raw)

# Only the excluded columns that really exist in raw
erased_in_raw = [c for c in ERASED_COLUMNS if c in raw.columns]

# Remaining columns (and candidate predictor columns excluding the target)
remaining_with_target = [c for c in raw.columns if c not in erased_in_raw]
final_features = [c for c in remaining_with_target if c != TARGET_COL]

assert TARGET_COL in remaining_with_target, f"'{TARGET_COL}' was dropped by mistake"

print(f"Initial columns: {len(raw.columns)}")
print(f"Excluded columns (found in raw): {len(erased_in_raw)} of {len(ERASED_COLUMNS)} listed")
print(f"Remaining columns (including '{TARGET_COL}'): {len(remaining_with_target)}")
print(f"Final candidate features for cleaning / PCA / RF (excluding '{TARGET_COL}'): {len(final_features)}\n")

for i, col in enumerate(final_features, start=1):
    raw_idx = raw.columns.index(col)
    print(f"{i:2d}. {col} (idx {raw_idx})")

Initial columns: 90
Excluded columns (found in raw): 36 of 36 listed
Remaining columns (including 'price'): 54
Final candidate features for cleaning / PCA / RF (excluding 'price'): 53

 1. hosts_time_as_user_years (idx 15)
 2. hosts_time_as_user_months (idx 16)
 3. hosts_time_as_host_years (idx 17)
 4. hosts_time_as_host_months (idx 18)
 5. host_location (idx 19)
 6. host_is_superhost (idx 24)
 7. host_listings_count (idx 28)
 8. host_has_profile_pic (idx 31)
 9. host_identity_verified (idx 32)
10. neighbourhood_cleansed (idx 34)
11. latitude (idx 36)
12. longitude (idx 37)
13. property_type (idx 38)
14. room_type (idx 39)
15. accommodates (idx 40)
16. bathrooms_text (idx 42)
17. bedrooms (idx 43)
18. beds (idx 44)
19. amenities (idx 45)
20. minimum_nights (idx 52)
21. maximum_nights (idx 53)
22. minimum_minimum_nights (idx 54)
23. maximum_minimum_nights (idx 55)
24. minimum_maximum_nights (idx 56)
25. maximum_maximum_nights (idx 57)
26. minimum_nights_avg_ntm (idx 58)
27. maximum_nigh

## Other outcome data that can affect the model

**estimated_occupancy_l365d** is a variable that can affect the model, because we are not measuring the demand for housing. So it is removed from the feature set: it captures a demand/occupancy outcome rather than an intrinsic property characteristic. Unless, it have low correlation with the price.

In [ ]:
OUTCOME_COLS = ["estimated_occupancy_l365d"]   # outcome, not a characteristic

In [14]:
from pyspark.sql import functions as F

df_corr = (
    raw.withColumn("price_num", F.regexp_replace("price", r"[$,€\s]", "").cast("double"))
       .withColumn("occ_num", F.col("estimated_occupancy_l365d").cast("double"))
       .filter((F.col("price_num") > 0) & F.col("occ_num").isNotNull())
       .withColumn("log_price", F.log("price_num"))
)

pearson_r = df_corr.stat.corr("log_price", "occ_num")
print(f"Real correlation beetween log(price) and estimated_occupancy_l365d: {pearson_r:.4f}")

Correlación real entre log(price) y estimated_occupancy_l365d: -0.0819


## 6. Quick look at the price (target)

In [15]:
priced = df.withColumn("price_num", F.regexp_replace("price", "[$,€\\s]", "").cast("double"))

n_price = priced.filter(F.col("price_num").isNotNull()).count()
print(f"Rows with a usable price: {n_price:,} of {n_rows:,} ({100 * n_price / n_rows:.1f}%)")

priced.select("price_num").summary("count", "min", "25%", "50%", "75%", "max").show()

q = priced.approxQuantile("price_num", [0.01, 0.05, 0.95, 0.99], 0.001)
print("Percentiles 1% / 5% / 95% / 99%:", q)

Rows with a usable price: 43,537 of 77,149 (56.4%)


+-------+---------+
|summary|price_num|
+-------+---------+
|  count|    43537|
|    min|    10.38|
|    25%|    139.0|
|    50%|    218.0|
|    75%|    359.0|
|    max| 100666.1|
+-------+---------+



Percentiles 1% / 5% / 95% / 99%: [46.28, 70.0, 849.55, 1786.67]


## 7. Categorical columns: how many values?

In [ ]:
# Categorical (text) candidates, taken from the final list
CATEGORICAL_COLS = [
    "room_type", "property_type", "neighbourhood_cleansed",
    "bathrooms_text", "host_location",
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "has_availability",
]

In [16]:

cat_cols = [c for c in CATEGORICAL_COLS if c in final_features]
skipped = [c for c in CATEGORICAL_COLS if c not in final_features]
if skipped:
    print("Not in final_features (skipped):", skipped)

# Only listings with a price: the rows we will really model
priced_df = raw.filter(
    F.col("price").isNotNull()
    & (F.trim(F.col("price")) != "")
    & (F.upper(F.trim(F.col("price"))) != "NULL")
).select(cat_cols).cache()

# Distinct counts for all columns in ONE Spark pass
distinct = priced_df.agg(*[F.countDistinct(c).alias(c) for c in cat_cols]).collect()[0]

for c in cat_cols:
    print(f"\n{c}: {distinct[c]} distinct values")
    priced_df.groupBy(c).count().orderBy(F.desc("count")).show(25, truncate=False)

priced_df.unpersist()


room_type: 4 distinct values
+---------------+-----+
|room_type      |count|
+---------------+-----+
|Entire home/apt|38131|
|Private room   |5010 |
|Hotel room     |278  |
|Shared room    |118  |
+---------------+-----+


property_type: 57 distinct values


+---------------------------------+-----+
|property_type                    |count|
+---------------------------------+-----+
|Entire rental unit               |35650|
|Private room in rental unit      |2491 |
|Room in hotel                    |1509 |
|Entire condo                     |1141 |
|Entire loft                      |485  |
|Private room in bed and breakfast|440  |
|Room in boutique hotel           |359  |
|Entire home                      |293  |
|Entire serviced apartment        |222  |
|Private room in condo            |166  |
|Entire townhouse                 |144  |
|Private room in home             |69   |
|Room in aparthotel               |68   |
|Shared room in rental unit       |61   |
|Private room in townhouse        |51   |
|Private room in hostel           |44   |
|Private room in loft             |36   |
|Shared room in hotel             |33   |
|Private room                     |26   |
|Private room in guest suite      |25   |
|Entire place                     

+-----------------+-----+
|bathrooms_text   |count|
+-----------------+-----+
|1 bath           |29347|
|2 baths          |3725 |
|1.5 baths        |3214 |
|1 private bath   |3029 |
|1 shared bath    |1270 |
|2.5 baths        |959  |
|3 baths          |714  |
|0 baths          |248  |
|3.5 baths        |173  |
|4 baths          |155  |
|Half-bath        |135  |
|1.5 shared baths |124  |
|Shared half-bath |99   |
|Private half-bath|69   |
|2 shared baths   |52   |
|4.5 baths        |51   |
|5 baths          |44   |
|NULL             |42   |
|0 shared baths   |39   |
|6 baths          |16   |
|5.5 baths        |9    |
|3 shared baths   |8    |
|6.5 baths        |4    |
|7 baths          |3    |
|10 baths         |1    |
+-----------------+-----+
only showing top 25 rows

host_location: 1075 distinct values
+----------------------------+-----+
|host_location               |count|
+----------------------------+-----+
|Paris, France               |28455|
|NULL                        |11227|

+--------------------+-----+
|host_has_profile_pic|count|
+--------------------+-----+
|t                   |41916|
|f                   |1621 |
+--------------------+-----+


host_identity_verified: 2 distinct values
+----------------------+-----+
|host_identity_verified|count|
+----------------------+-----+
|t                     |43231|
|f                     |306  |
+----------------------+-----+


has_availability: 2 distinct values
+----------------+-----+
|has_availability|count|
+----------------+-----+
|t               |42620|
|f               |908  |
|NULL            |9    |
+----------------+-----+



DataFrame[room_type: string, property_type: string, neighbourhood_cleansed: string, bathrooms_text: string, host_location: string, host_is_superhost: string, host_has_profile_pic: string, host_identity_verified: string, has_availability: string]

## 8. Create the 5,000-row sample
The kept columns are still raw (not cleaned), so the same cleaning code works on both files.
Only rows **with a price** are sampled: rows without the target are dropped by cleaning anyway
(43.6% of the raw file), so keeping them would waste almost half of the sample.
Written with pandas to avoid Spark's winutils problem on Windows.

In [24]:
# Columns for the sample: the 52 candidate features + the target (price)
OUTCOME_COLS = ["estimated_occupancy_l365d"]   # outcome, not a characteristic (see reduction_variables.md)
FEATURE_COLS = [c for c in final_features if c not in OUTCOME_COLS]
SAMPLE_COLS = [TARGET_COL] + FEATURE_COLS

print(f"Features: {len(FEATURE_COLS)} | Sample columns (with '{TARGET_COL}'): {len(SAMPLE_COLS)}")
assert len(FEATURE_COLS) == 52, f"Expected 52 features, got {len(FEATURE_COLS)}"

# Filter the price => keep only rows with a price that is not null, "" or "NULL"
price_ok = (
    F.col(TARGET_COL).isNotNull()
    & (F.trim(F.col(TARGET_COL)) != "")
    & (F.upper(F.trim(F.col(TARGET_COL))) != "NULL")
)
priced_raw = raw.filter(price_ok).select(SAMPLE_COLS)

n_priced = priced_raw.count()
assert n_priced >= SAMPLE_SIZE, f"Only {n_priced:,} priced rows, less than {SAMPLE_SIZE:,}"

# Uniform random sample of exactly SAMPLE_SIZE rows (reproducible with the seed)
shuffled = priced_raw.orderBy(F.rand(seed=SEED))
sample_pdf = shuffled.limit(SAMPLE_SIZE).toPandas()

# Create the folder (if needed) and save the file
os.makedirs(SAMPLE_DIR, exist_ok=True)
sample_path = os.path.join(SAMPLE_DIR, "listings_sample.csv")
sample_pdf.to_csv(sample_path, index=False)
print(f"Sample written: {sample_path} ({len(sample_pdf):,} of {n_priced:,} priced rows)")

# Check it reads back with the same options as the full file
reader = spark.read.option("header", True).option("multiLine", True).option("escape", '"')
check = reader.csv(sample_path)
n_check = check.count()
print("Re-read:", n_check, "rows,", len(check.columns), "columns")
assert n_check == SAMPLE_SIZE
assert len(check.columns) == len(SAMPLE_COLS), "Column count changed when re-reading"


# Create the folder (if needed) and save the file
os.makedirs(SAMPLE_DIR, exist_ok=True)
sample_path = os.path.join(SAMPLE_DIR, "listings_sample.csv")
sample_pdf.to_csv(sample_path, index=False)
print(f"Sample written: {sample_path} ({len(sample_pdf):,} of {n_priced:,} priced rows)")

# Check it reads back with the same options as the full file
reader = spark.read.option("header", True).option("multiLine", True).option("escape", '"')
check = reader.csv(sample_path)
n_check = check.count()
print("Re-read:", n_check, "rows,", len(check.columns), "columns")
assert n_check == SAMPLE_SIZE
assert len(check.columns) == len(SAMPLE_COLS), "Column count changed when re-reading"

Features: 52 | Sample columns (with 'price'): 53


Sample written: ../../data/sample/listings_sample.csv (5,000 of 43,537 priced rows)
Re-read: 5000 rows, 53 columns
Sample written: ../../data/sample/listings_sample.csv (5,000 of 43,537 priced rows)
Re-read: 5000 rows, 53 columns
